# Projeto A3: Mobilidade urbana em Salvador
## Notebook 1: ETL (Extração, Transformação e Carga)

**UC:** Análise de Dados e Big Data (UNIFACS) · **Professor:** Eduardo Sidney da Silva Xavier
**Tema:** Mobilidade urbana: transporte público e o crescimento da população em Salvador-BA

**Equipe:** Anthony Natan Santos Pereira · Caio Branco Marques · Clara Gil Moreira de Alencar Santiago ·
Fernanda Santos da Silva · Lukas Carneiro Rocha

---
Este notebook transforma arquivos brutos de **fontes oficiais** em uma base analítica integrada,
carregada em um banco **SQLite** e exportada em **CSV**.

| Etapa | O que acontece | Onde |
|---|---|---|
| **E**xtração | Download dos arquivos oficiais e leitura de documentos oficiais | `scripts/coleta_dados_brutos.py` e seção 1 |
| **T**ransformação | Limpeza, padronização, compatibilização da série de população, deflação da tarifa, integração e criação de indicadores | seções 2 a 9 |
| **C**arga | Gravação das tabelas no banco `data/db/mobilidade_salvador.sqlite` e em `data/processed/*.csv` | seção 10 |

Cada decisão de tratamento está explicada em uma célula de texto antes do código que a executa.

In [1]:
# ============================================================
# 0. CONFIGURAÇÃO DO AMBIENTE
# ============================================================
# pandas: tabelas | numpy: contas | sqlite3: banco de dados (já vem com o Python)
# xlrd/openpyxl: planilhas do Excel | pdfplumber: texto de PDF | libarchive: arquivo .rar
import hashlib
import json
import os
import re
import sqlite3
import subprocess
import sys
import unicodedata
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

# >>> No Google Colab: troque o endereço abaixo pelo do repositório da equipe no GitHub. <<<
REPO_GITHUB = "https://github.com/SEU-USUARIO/mobilidade-salvador-a3"
warnings.filterwarnings("ignore")  # avisos de formatação das planilhas do governo, sem efeito nos dados
NO_COLAB = "google.colab" in sys.modules
if NO_COLAB:  # no Colab: instala o que falta e baixa o repositório (que já traz os dados brutos)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pdfplumber", "xlrd", "libarchive-c"], check=False)
    subprocess.run(["apt-get", "-qq", "install", "-y", "libarchive-dev"], check=False)
    if not Path("/content/projeto").exists():
        subprocess.run(["git", "clone", "-q", REPO_GITHUB, "/content/projeto"], check=True)
    os.chdir("/content/projeto")

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BRUTOS = RAIZ / "data" / "raw"
PROCESSADOS = RAIZ / "data" / "processed"
BANCO = RAIZ / "data" / "db" / "mobilidade_salvador.sqlite"
for p in (PROCESSADOS, BANCO.parent):
    p.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

## 1. Extração

### 1.1 Fontes e métodos de extração

| # | Fonte (órgão) | Conteúdo | Método de extração | Arquivo bruto em `data/raw/` |
|---|---|---|---|---|
| 1 | **IBGE**: Estimativas da população (tab. 6579) | População 2001–2026 das 13 capitais | API SIDRA (JSON) | `ibge/sidra_6579_*.json` |
| 2 | **IBGE**: Censo 2022 (tab. 4709 e 4714) | População 2010 e 2022, área e densidade | API SIDRA (JSON) | `ibge/sidra_4709_*.json`, `ibge/sidra_4714_*.json` |
| 3 | **IBGE**: Censo 2022 (tab. 10330, 10332, 10333) e Censo 2010 (tab. 3604) | Meio de transporte e tempo casa-trabalho | API SIDRA + metadados (JSON) | `ibge/sidra_10330_*.json` etc. |
| 4 | **IBGE**: IPCA (tab. 1737) | Número-índice mensal da inflação | API SIDRA (JSON) | `ibge/sidra_1737_ipca_indice.json` |
| 5 | **SENATRAN / Ministério dos Transportes** | Frota por município e tipo, dezembro de 2014 a 2025 | Planilhas XLS/XLSX/RAR do portal gov.br | `senatran/frota_municipio_tipo_dez_AAAA.*` |
| 6 | **Ministério das Cidades: PEMOB** | Frota operacional, passageiros, km rodados e velocidade dos ônibus (2022 e 2025) | Planilhas XLSX | `pemob/*.xlsx` |
| 7 | **SEI-BA (Governo da Bahia)**: IMEC | Índice mensal de passageiros de ônibus urbanos de Salvador | Tabela em PDF | `sei/TABELAS_IMEC_DEZEMBRO_2025.pdf` |
| 8 | **CTB (Governo da Bahia)** | Passageiros mensais do metrô 2024–2026 | Tabela do demonstrativo oficial (imagem) transcrita e validada | `ctb/metro_passageiros_mensal_ctb.csv` |
| 9 | **Prefeitura de Salvador/SEMOB, ARSAL, Cia. do Metrô da Bahia, Integra** | Tarifas, linhas, passageiros do metrô por ano e indicadores do STCO | Leitura de documentos oficiais → tabela com fonte, trecho e precisão | `documentos/fontes_documentais.csv` |

O script `scripts/coleta_dados_brutos.py` faz o download automático (fontes 1–7). Arquivos que o servidor
recusou a partir do Colab (servidores do governo que bloqueiam acessos de fora do Brasil) foram baixados
pelo navegador, e a integridade foi conferida pelo **hash SHA-256** de cada arquivo (seção 1.3).

In [2]:
# Se a pasta de dados brutos estiver vazia (ex.: rodando pela primeira vez), executa a coleta automática.
if not any((BRUTOS / "senatran").glob("frota_*")):
    subprocess.run([sys.executable, str(RAIZ / "scripts" / "coleta_dados_brutos.py")],
                   env={**os.environ, "PASTA_DADOS_BRUTOS": str(BRUTOS)}, check=False)

inventario = pd.DataFrame([
    {"pasta": p.parent.name, "arquivo": p.name, "kb": round(p.stat().st_size / 1024, 1)}
    for p in sorted(BRUTOS.rglob("*")) if p.is_file()
])
print(f"{len(inventario)} arquivos brutos")
inventario

38 arquivos brutos


,pasta,arquivo,kb
0,ctb,Passageiros_Transportados_SET_26.jpg,77.3
1,ctb,metro_passageiros_mensal_ctb.csv,0.5
2,documentos,fontes_documentais.csv,13.3
3,ibge,meta_10330.json,3.9
4,ibge,meta_10332.json,3.0
5,ibge,meta_10333.json,2.9
6,ibge,meta_3422.json,1.5
7,ibge,meta_3604.json,2.9
8,ibge,sidra_10330_censo2022_deslocamento.json,2003.1
9,ibge,sidra_10332_censo2022_deslocamento.json,201.6


### 1.2 Funções de leitura
**API SIDRA (IBGE):** a resposta é uma lista JSON cujo **primeiro item é o cabeçalho** (ex.: `"D1N": "Município"`).
A função usa esse cabeçalho para nomear as colunas e converte o valor em número; os símbolos especiais do IBGE
(`-`, `...`, `X`) viram dado ausente (`NaN`).

In [3]:
def ler_sidra(caminho):
    """Lê um JSON da API SIDRA e devolve uma tabela arrumada (uma linha por valor)."""
    with open(caminho, encoding="utf-8") as f:
        dados = json.load(f)
    df = pd.DataFrame(dados[1:]).rename(columns=dados[0])
    df["Valor"] = pd.to_numeric(df["Valor"], errors="coerce")
    if "Município (Código)" in df:
        df["cod_ibge"] = df["Município (Código)"].astype(int)
    return df


def sem_acento(txt):
    """Padroniza textos: maiúsculas, sem acentos e sem espaços duplicados."""
    txt = unicodedata.normalize("NFKD", str(txt)).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", txt).strip().upper()


# Salvador + as 12 outras capitais com mais de 1 milhão de habitantes no Censo 2022 (grupo de comparação)
CAPITAIS = pd.DataFrame([
    (2927408, "Salvador", "BA", "Nordeste"), (3550308, "São Paulo", "SP", "Sudeste"),
    (3304557, "Rio de Janeiro", "RJ", "Sudeste"), (5300108, "Brasília", "DF", "Centro-Oeste"),
    (2304400, "Fortaleza", "CE", "Nordeste"), (3106200, "Belo Horizonte", "MG", "Sudeste"),
    (1302603, "Manaus", "AM", "Norte"), (4106902, "Curitiba", "PR", "Sul"),
    (2611606, "Recife", "PE", "Nordeste"), (5208707, "Goiânia", "GO", "Centro-Oeste"),
    (1501402, "Belém", "PA", "Norte"), (4314902, "Porto Alegre", "RS", "Sul"),
    (2111300, "São Luís", "MA", "Nordeste"),
], columns=["cod_ibge", "municipio", "uf", "regiao"])
CAPITAIS["chave"] = CAPITAIS["uf"] + "|" + CAPITAIS["municipio"].map(sem_acento)
SALVADOR = 2927408
CAPITAIS

,cod_ibge,municipio,uf,regiao,chave
0,2927408,Salvador,BA,Nordeste,BA|SALVADOR
1,3550308,São Paulo,SP,Sudeste,SP|SAO PAULO
2,3304557,Rio de Janeiro,RJ,Sudeste,RJ|RIO DE JANEIRO
3,5300108,Brasília,DF,Centro-Oeste,DF|BRASILIA
4,2304400,Fortaleza,CE,Nordeste,CE|FORTALEZA
5,3106200,Belo Horizonte,MG,Sudeste,MG|BELO HORIZONTE
6,1302603,Manaus,AM,Norte,AM|MANAUS
7,4106902,Curitiba,PR,Sul,PR|CURITIBA
8,2611606,Recife,PE,Nordeste,PE|RECIFE
9,5208707,Goiânia,GO,Centro-Oeste,GO|GOIANIA


### 1.3 Integridade dos arquivos baixados pelo navegador
Os arquivos da SENATRAN de 2017–2019 e 2021–2023 foram baixados pelo navegador. No momento do download foi
calculado o SHA-256 de cada arquivo e registrado em `senatran_extracao_navegador_2017_2023.json`. Aqui o
hash é recalculado: se for igual, o arquivo é **idêntico** ao publicado pelo governo.

In [4]:
reg = json.load(open(BRUTOS / "senatran" / "senatran_extracao_navegador_2017_2023.json", encoding="utf-8"))
checagem = []
for ano, info in reg.items():
    arq = BRUTOS / "senatran" / f"frota_municipio_tipo_dez_{ano}.xls"
    h = hashlib.sha256(arq.read_bytes()).hexdigest()
    checagem.append({"ano": int(ano), "url_oficial": info["url"], "sha256_confere": h == info["sha256"],
                     "total_nacional_publicado": info.get("totalNacional", info.get("total"))})
pd.DataFrame(checagem)

,ano,url_oficial,sha256_confere,total_nacional_publicado
0,2017,https://www.gov.br/transportes/pt-br/assuntos/...,True,97091956
1,2018,https://www.gov.br/transportes/pt-br/assuntos/...,True,100746553
2,2019,https://www.gov.br/transportes/pt-br/assuntos/...,True,104784375
3,2021,https://www.gov.br/transportes/pt-br/assuntos/...,True,111446870
4,2022,https://www.gov.br/transportes/pt-br/assuntos/...,True,115116532
5,2023,https://www.gov.br/transportes/pt-br/assuntos/...,True,119227657


## 2. População (IBGE)

**Problema de qualidade encontrado.** A série oficial de estimativas do IBGE **não é comparável no tempo**:
* as estimativas de 2011 a 2021 partiram do Censo 2010 e **superestimaram** a população (em Salvador, a
  estimativa de 2021 era 2,90 milhões, mas o Censo 2022 contou 2,42 milhões);
* as estimativas a partir de 2024 partem do Censo 2022 **ajustado pela cobertura** (correção de pessoas não
  recenseadas), o que eleva o nível em cerca de 6% em relação à contagem do Censo.

Usar a série "crua" criaria saltos artificiais nos indicadores por habitante em 2022 e 2024.

**Decisão (série compatibilizada `populacao`):**
1. **2010 e 2022:** contagens dos Censos (tab. 4709; 2010 = população 2022 − variação absoluta, já nos limites
   municipais de 2022);
2. **2011–2021:** interpolação geométrica entre os Censos: $P_t = P_{2010}\,(P_{2022}/P_{2010})^{(t-2010)/12}$;
3. **2023–2026:** parte-se do Censo 2022 e aplica-se a **taxa de variação anual** das estimativas oficiais
   recentes (2025/2024 e 2026/2025; para 2023 e 2024 usa-se 2025/2024).

A estimativa oficial fica preservada na coluna `pop_estimativa_oficial`.

In [5]:
est = ler_sidra(BRUTOS / "ibge" / "sidra_6579_estimativas_populacao.json")
est = est.assign(ano=est["Ano"].astype(int))[["cod_ibge", "ano", "Valor"]].rename(columns={"Valor": "pop_estimativa_oficial"})

censo = ler_sidra(BRUTOS / "ibge" / "sidra_4709_censo2022_variacao2010.json")
c22 = censo[censo["Variável"].str.startswith("População residente")].set_index("cod_ibge")["Valor"]
var = censo[censo["Variável"].str.startswith("Variação absoluta")].set_index("cod_ibge")["Valor"]
censos = pd.DataFrame({"censo_2010": c22 - var, "censo_2022": c22}).reset_index()
censos["var_pct_2010_2022"] = (censos["censo_2022"] / censos["censo_2010"] - 1) * 100
censos.merge(CAPITAIS[["cod_ibge", "municipio"]]).sort_values("var_pct_2010_2022")

,cod_ibge,censo_2010,censo_2022,var_pct_2010_2022,municipio
0,2927408,2675656.0,2417678.0,-9.641673,Salvador
10,1501402,1393399.0,1303403.0,-6.458739,Belém
11,4314902,1409351.0,1332845.0,-5.428456,Porto Alegre
8,2611606,1537704.0,1488920.0,-3.172522,Recife
5,3106200,2375609.0,2315560.0,-2.527731,Belo Horizonte
2,3304557,6320446.0,6211223.0,-1.728090,Rio de Janeiro
4,2304400,2459712.0,2428708.0,-1.260473,Fortaleza
7,4106902,1751907.0,1773718.0,1.244986,Curitiba
1,3550308,11253503.0,11451999.0,1.763860,São Paulo
12,2111300,1015329.0,1037775.0,2.210712,São Luís


In [6]:
def serie_populacao(cod):
    c = censos.set_index("cod_ibge").loc[cod]
    e = est[est["cod_ibge"] == cod].set_index("ano")["pop_estimativa_oficial"]
    pop = {t: c["censo_2010"] * (c["censo_2022"] / c["censo_2010"]) ** ((t - 2010) / 12) for t in range(2010, 2023)}
    taxa = e.loc[2025] / e.loc[2024]
    taxas = {2023: taxa, 2024: taxa, 2025: taxa, 2026: e.loc[2026] / e.loc[2025]}
    for t in range(2023, 2027):
        pop[t] = pop[t - 1] * taxas[t]
    out = pd.DataFrame({"ano": list(pop), "populacao": np.round(list(pop.values())).astype("int64")})
    out["cod_ibge"] = cod
    out["metodo_pop"] = np.select([out["ano"].isin([2010, 2022]), out["ano"] < 2022],
                                  ["censo", "interpolacao_intercensitaria"], "censo2022_x_taxa_estimativas")
    return out


populacao = pd.concat([serie_populacao(c) for c in CAPITAIS["cod_ibge"]], ignore_index=True)
populacao = populacao.merge(est, on=["cod_ibge", "ano"], how="left")
populacao[populacao["cod_ibge"] == SALVADOR]

,ano,populacao,cod_ibge,metodo_pop,pop_estimativa_oficial
0,2010,2675656,2927408,censo,NaN
1,2011,2653145,2927408,interpolacao_intercensitaria,2693606.0
2,2012,2630823,2927408,interpolacao_intercensitaria,2710968.0
3,2013,2608689,2927408,interpolacao_intercensitaria,2883682.0
4,2014,2586741,2927408,interpolacao_intercensitaria,2902927.0
5,2015,2564978,2927408,interpolacao_intercensitaria,2921087.0
6,2016,2543398,2927408,interpolacao_intercensitaria,2938092.0
7,2017,2522000,2927408,interpolacao_intercensitaria,2953986.0
8,2018,2500781,2927408,interpolacao_intercensitaria,2857329.0
9,2019,2479742,2927408,interpolacao_intercensitaria,2872347.0


In [7]:
dens = ler_sidra(BRUTOS / "ibge" / "sidra_4714_censo2022_densidade.json")
territorio = (dens.pivot_table(index="cod_ibge", columns="Variável", values="Valor").reset_index())
territorio.columns.name = None
territorio = territorio.rename(columns=lambda c: "area_km2" if "Área" in c else ("densidade_2022" if "Densidade" in c else c))
territorio = territorio[["cod_ibge", "area_km2", "densidade_2022"]]
territorio.merge(CAPITAIS[["cod_ibge", "municipio"]])

,cod_ibge,area_km2,densidade_2022,municipio
0,1302603,11401.092,181.01,Manaus
1,1501402,1059.466,1230.25,Belém
2,2111300,583.063,1779.87,São Luís
3,2304400,312.353,7775.52,Fortaleza
4,2611606,218.843,6803.60,Recife
5,2927408,693.442,3486.49,Salvador
6,3106200,331.354,6988.18,Belo Horizonte
7,3304557,1200.329,5174.60,Rio de Janeiro
8,3550308,1521.202,7528.26,São Paulo
9,4106902,434.892,4078.53,Curitiba


## 3. Frota de veículos (SENATRAN)

Usamos o mês de **dezembro** (estoque de fim de ano) de **2014 a 2025**. Antes de 2014 o portal não publica a
tabela "frota por município e tipo"; os anos anteriores só existem como serviço contratado ao SERPRO ou em
arquivos de outro formato (marca/modelo), por isso a série começa em 2014, ano da chegada do metrô.

**Problemas tratados:**
* **formatos diferentes** (XLS, XLSX e RAR) → a função detecta o formato e descompacta o RAR;
* **título e linhas extras acima da tabela** e cabeçalho repetido → localizamos a linha que contém "MUNICIPIO";
* planilha de **glossário** na primeira aba (2014 e 2015) → usamos a aba que contém a tabela;
* aba de 2015 nomeada `JUL_2015`, mas com título "DEZ/2015" → conferido pelo título, mantida;
* os arquivos **não têm código IBGE** → cruzamento por UF + nome padronizado.

**Validação:** a soma de todos os municípios precisa bater com o total nacional impresso no topo da planilha.

**Agrupamento dos tipos (decisão analítica):**
* `autos` = AUTOMÓVEL + CAMINHONETE + CAMIONETA + UTILITÁRIO (veículos leves);
* `motos` = MOTOCICLETA + MOTONETA + CICLOMOTOR;
* `onibus_registrados` = ÔNIBUS + MICRO-ÔNIBUS emplacados no município (inclui fretamento/turismo/intermunicipais,
  por isso **não** é a frota do sistema municipal, que vem da PEMOB).

In [8]:
def abrir_planilha(caminho):
    caminho = Path(caminho)
    if caminho.suffix.lower() == ".rar":
        import libarchive
        destino = caminho.with_name(caminho.stem + "_extraido")
        destino.mkdir(exist_ok=True)
        atual = os.getcwd()
        os.chdir(destino)
        libarchive.extract_file(str(caminho))
        os.chdir(atual)
        caminho = next(p for p in destino.rglob("*") if p.suffix.lower() in (".xls", ".xlsx"))
    abas = pd.read_excel(caminho, sheet_name=None, header=None, dtype=object)
    for nome, df in abas.items():
        txt = df.astype(str).map(sem_acento)
        if (txt == "MUNICIPIO").any().any():
            return nome, df
    raise ValueError(f"tabela não encontrada em {caminho.name}")


def ler_frota(caminho, ano):
    aba, bruto = abrir_planilha(caminho)
    linha_cab = bruto.index[bruto.astype(str).map(sem_acento).eq("MUNICIPIO").any(axis=1)][0]
    total_publicado = pd.to_numeric(bruto.iloc[:linha_cab].stack(), errors="coerce").max()
    df = bruto.iloc[linha_cab + 1:].copy()
    df.columns = [sem_acento(c).replace("-", " ") for c in bruto.iloc[linha_cab]]
    df = df[df["MUNICIPIO"].notna() & (df["MUNICIPIO"].map(sem_acento) != "MUNICIPIO")]
    num = lambda col: pd.to_numeric(df[col], errors="coerce").fillna(0) if col in df else 0
    soma_municipios = num("TOTAL").sum()
    df["chave"] = df["UF"].map(sem_acento) + "|" + df["MUNICIPIO"].map(sem_acento)
    out = pd.DataFrame({
        "chave": df["chave"], "ano": ano, "frota_total": num("TOTAL"), "automovel": num("AUTOMOVEL"),
        "autos": num("AUTOMOVEL") + num("CAMINHONETE") + num("CAMIONETA") + num("UTILITARIO"),
        "motos": num("MOTOCICLETA") + num("MOTONETA") + num("CICLOMOTOR"),
        "onibus_registrados": num("ONIBUS") + num("MICRO ONIBUS"),
    })
    out = out[out["chave"].isin(CAPITAIS["chave"])]
    validacao = {"ano": ano, "aba": aba, "total_publicado": total_publicado, "soma_municipios": soma_municipios,
                 "confere": abs(total_publicado - soma_municipios) < 1, "capitais_encontradas": len(out)}
    return out, validacao


frotas, validacoes = [], []
for arq in sorted((BRUTOS / "senatran").glob("frota_municipio_tipo_dez_*")):
    if arq.is_dir() or arq.suffix.lower() not in (".xls", ".xlsx", ".rar"):
        continue
    ano = int(re.search(r"(\d{4})", arq.name).group(1))
    f, v = ler_frota(arq, ano)
    frotas.append(f)
    validacoes.append(v)
frota = (pd.concat(frotas, ignore_index=True).merge(CAPITAIS[["chave", "cod_ibge"]], on="chave")
         .drop(columns="chave").sort_values(["cod_ibge", "ano"]).reset_index(drop=True))
for c in ["frota_total", "automovel", "autos", "motos", "onibus_registrados"]:
    frota[c] = frota[c].astype("int64")
pd.DataFrame(validacoes)

,ano,aba,total_publicado,soma_municipios,confere,capitais_encontradas
0,2014,DEZ_2014,86700490.0,86700490,True,13
1,2015,JUL_2015,90686936.0,90686936,True,13
2,2016,DEZ_2016,93867016.0,93867016,True,13
3,2017,DEZ_2017,97091956.0,97091956,True,13
4,2018,DEZ 2018,100746553.0,100746553,True,13
5,2019,Dezembro 2019,104784375.0,104784375,True,13
6,2020,DEZEMBRO_2020,107948371.0,107948371,True,13
7,2021,DEZEMBRO_2021,111446870.0,111446870,True,13
8,2022,DEZEMBRO_2022,115116532.0,115116532,True,13
9,2023,DEZEMBRO_2023,119227657.0,119227657,True,13


In [9]:
frota[frota["cod_ibge"] == SALVADOR]

,ano,frota_total,automovel,autos,motos,onibus_registrados,cod_ibge
60,2014,819237,551687,655553,119584,13016,2927408
61,2015,846102,564606,674223,126513,13638,2927408
62,2016,866032,573138,688002,132216,13478,2927408
63,2017,886703,583805,703215,137745,13126,2927408
64,2018,915186,597610,723830,145179,12796,2927408
65,2019,945305,610461,743710,153767,13160,2927408
66,2020,964263,615701,753288,161491,13556,2927408
67,2021,982619,619056,760993,170329,14062,2927408
68,2022,1006292,623395,771509,181511,14512,2927408
69,2023,1036885,630657,786187,194844,14953,2927408


## 4. Inflação (IPCA) e tarifa real
A tarifa nominal subiu de R$ 2,30 (2010) para R$ 5,90 (2026), mas parte disso é inflação. Deflacionamos pelo IPCA
(IBGE) para reais de **dezembro de 2025**:

$$\text{tarifa real}_t = \text{tarifa média}_t \times \frac{\text{IPCA}_{dez/2025}}{\text{IPCA médio}_t}$$

A **tarifa média do ano** pondera os dias de vigência de cada valor (ex.: 2022 teve R$ 4,40 até 3/jun e
R$ 4,90 depois). Datas de vigência: comunicados oficiais da Prefeitura/SEMOB e ARSAL (`fontes_documentais.csv`).

In [10]:
ipca = ler_sidra(BRUTOS / "ibge" / "sidra_1737_ipca_indice.json")
ipca = pd.DataFrame({"data": pd.to_datetime(ipca["Mês (Código)"], format="%Y%m"), "ipca_indice": ipca["Valor"]})
ipca["ano"] = ipca["data"].dt.year
ipca_anual = ipca.groupby("ano", as_index=False)["ipca_indice"].mean().rename(columns={"ipca_indice": "ipca_medio_ano"})
IPCA_BASE = ipca.loc[ipca["data"] == "2025-12-01", "ipca_indice"].iloc[0]

docs = pd.read_csv(BRUTOS / "documentos" / "fontes_documentais.csv")

# Correção documentada: o balanço anual da CTB e da concessionária (divulgado em 21/01/2025) traz os números
# EXATOS de 2019 e 2023. Eles substituem os valores arredondados registrados antes ("106 milhões" e
# "cerca de 102 milhões"). O arquivo bruto não é alterado; a troca acontece aqui, na transformação.
BALANCO_CTB_2025 = "https://atarde.com.br/salvador/metro-supera-117-milhoes-de-usuarios-em-2024-e-bate-recorde-1304059"
for ano_corr, valor_exato in [(2019, 107_431_208), (2023, 101_684_054)]:
    linha = (docs["tabela"] == "metro") & (docs["indicador"] == "passageiros_ano") & (docs["ano"] == ano_corr)
    docs.loc[linha, ["valor", "precisao", "url", "data_acesso"]] = [valor_exato, "exato (corrigido na transformação)",
                                                                    BALANCO_CTB_2025, "2026-11-17"]
    docs.loc[linha, ["orgao_fonte", "documento"]] = ["Companhia de Transportes do Estado da Bahia (CTB) e CCR Metro Bahia",
                                                     "Balanco anual divulgado pela CTB e concessionaria (21/01/2025)"]

tar = docs[docs["tabela"] == "tarifa"].copy()
tar["vigencia"] = pd.to_datetime(tar["observacao"].str.extract(r"vigencia=(\d{4}-\d{2}-\d{2})")[0])
tar = tar.sort_values("vigencia")[["vigencia", "valor"]].rename(columns={"valor": "tarifa"})
dias = pd.DataFrame({"data": pd.date_range("2010-01-01", "2026-12-31", freq="D")})
dias = pd.merge_asof(dias, tar, left_on="data", right_on="vigencia")
dias["tarifa"] = dias["tarifa"].fillna(2.20)  # antes de 16/01/2010 vigorava R$ 2,20
dias["ano"] = dias["data"].dt.year
tarifa = (dias.groupby("ano").agg(tarifa_media_ano=("tarifa", "mean"), tarifa_dezembro=("tarifa", "last"))
          .reset_index().merge(ipca_anual, on="ano", how="left"))
tarifa["tarifa_real_dez2025"] = tarifa["tarifa_media_ano"] * IPCA_BASE / tarifa["ipca_medio_ano"]
tarifa = tarifa[tarifa["ano"] <= 2025]
tarifa.round(2)

,ano,tarifa_media_ano,tarifa_dezembro,ipca_medio_ano,tarifa_real_dez2025
0,2010,2.30,2.3,3114.50,5.46
1,2011,2.50,2.5,3321.20,5.57
2,2012,2.67,2.8,3500.66,5.65
3,2013,2.80,2.8,3717.85,5.58
4,2014,2.80,2.8,3953.15,5.24
5,2015,3.00,3.0,4310.12,5.15
6,2016,3.30,3.3,4686.79,5.21
7,2017,3.60,3.6,4848.31,5.50
8,2018,3.70,3.7,5025.99,5.45
9,2019,4.00,4.0,5213.61,5.68


## 5. Metrô (CTB e Companhia do Metrô da Bahia)
* **Anual (2018–2025):** relatórios da administração das demonstrações financeiras auditadas da Companhia do
  Metrô da Bahia e balanços da CTB (`fontes_documentais.csv`). Alguns valores são **arredondados pela própria
  fonte** ("quase 90 milhões"), e a coluna `precisao` registra isso. Para 2019, 2023 e 2024 usamos os números
  exatos do balanço CTB/concessionária de jan/2025 (a troca de 2019 e 2023 está na seção 4, logo após a leitura).
* **Mensal (jan/2024–set/2026):** demonstrativo oficial da CTB, publicado como imagem com a tabela de valores;
  os números foram transcritos e **validados**: a soma de 2024 dá 117.559.289 (CTB/CCR divulgaram 117.533.257;
  diferença de 0,02%) e a de 2025 dá 117.507.964 (divulgado: 117,5 milhões).

In [11]:
metro_mensal = pd.read_csv(BRUTOS / "ctb" / "metro_passageiros_mensal_ctb.csv")
metro_mensal["data"] = pd.to_datetime(dict(year=metro_mensal["ano"], month=metro_mensal["mes"], day=1))
validacao_metro = metro_mensal.groupby("ano")["passageiros"].agg(["sum", "count"]).rename(columns={"sum": "soma_meses", "count": "meses"})
metro_anual = docs[(docs["tabela"] == "metro") & (docs["indicador"] == "passageiros_ano")][["ano", "valor", "precisao", "orgao_fonte"]]
metro_anual = metro_anual.rename(columns={"valor": "metro_passageiros_ano", "precisao": "metro_precisao", "orgao_fonte": "metro_fonte"})
display(validacao_metro)
metro_anual

,soma_meses,meses
ano,,
2024,117559289,12
2025,117507964,12
2026,88120603,9


,ano,metro_passageiros_ano,metro_precisao,metro_fonte
0,2018,90000000.0,aproximado (quase 90 milhoes),Companhia do Metro da Bahia (concessionaria CC...
1,2019,107431208.0,exato (corrigido na transformação),Companhia de Transportes do Estado da Bahia (C...
2,2020,62000000.0,aproximado (quase 62 milhoes),Companhia do Metro da Bahia (concessionaria CC...
3,2021,72000000.0,aproximado (quase 72 milhoes),Companhia do Metro da Bahia (concessionaria CC...
4,2022,95000000.0,aproximado (cerca de 95 milhoes),Companhia do Metro da Bahia (concessionaria CC...
5,2023,101684054.0,exato (corrigido na transformação),Companhia de Transportes do Estado da Bahia (C...
6,2024,117533257.0,exato,Companhia de Transportes do Estado da Bahia (C...
7,2025,117500000.0,aproximado (117.5 milhoes),Motiva / CCR Metro Bahia (concessionaria)


## 6. Sistema de ônibus de Salvador (STCO): PEMOB e documentos oficiais

Salvador **não publica uma série aberta** de passageiros e frota do STCO (diferente de Belo Horizonte, por exemplo,
que mantém dados abertos por linha). Montamos a série a partir de:
* **PEMOB** (Ministério das Cidades): respostas oficiais da Prefeitura sobre 2022 (edição 2023) e 2025 (edição 2026):
  frota operacional, km rodados, passageiros, velocidade média, idade da frota, viagens não completadas;
* **SEMOB**: passageiros por dia em 2019 e 2024 (documento oficial de out/2025);
* **Relatórios de Atividades da Prefeitura**: linhas (2017–2018) e passageiros/mês (2018);
* **Integra Salvador** (concessionárias): retrato atual (linhas, frota diária, km/mês), sem data → usado só como
  referência do presente.

**Cuidado com definições:** "passageiros equivalentes" (PEMOB) = receita ÷ tarifa, o que confere em 2022:
R$ 881,6 milhões ÷ R$ 4,90 ≈ 179,9 milhões. É a medida mais estável entre as edições; as demais categorias
(comuns, vale-transporte etc.) mudaram de definição entre 2023 e 2026 e **não** são somadas.

In [12]:
CAMPOS_PEMOB = {
    "1.5.1.A": "frota_onibus_convencional", "1.5.1.E": "frota_onibus_articulado", "1.5.1.I": "frota_onibus_biarticulado",
    "1.5.1.M": "frota_micro_vans", "1.7.1.A": "km_onibus_ano", "2.1.1.A": "velocidade_brt_pico",
    "2.1.1.B": "velocidade_corredores_pico", "2.1.2": "velocidade_trafego_misto_pico", "2.3.1.A": "idade_media_frota",
    "2.6.2.I": "viagens_nao_completadas", "3.1.1.A": "tarifa_pemob", "3.2.1.A": "receita_tarifaria_onibus",
    "3.4.1.F": "passageiros_equivalentes_onibus",
}


def ler_pemob(caminho, ano_referencia):
    d = pd.read_excel(caminho, header=None, dtype=str)
    codigos = d.iloc[1].astype(str).str.strip()
    corpo = d.iloc[2:].copy()
    corpo.columns = codigos
    corpo = corpo[corpo["CÓDIGO IBGE"].astype(str).str.strip().str.isdigit()]
    corpo["cod_ibge"] = corpo["CÓDIGO IBGE"].astype(int)
    corpo = corpo[corpo["cod_ibge"].isin(CAPITAIS["cod_ibge"])]
    out = corpo[["cod_ibge"] + [c for c in CAMPOS_PEMOB if c in corpo]].rename(columns=CAMPOS_PEMOB)
    for c in out.columns[1:]:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    out["ano"] = ano_referencia
    out["edicao_pemob"] = ano_referencia + 1
    return out


pemob = pd.concat([ler_pemob(BRUTOS / "pemob" / "pemob2023_municipal.xlsx", 2022),
                   ler_pemob(BRUTOS / "pemob" / "pemob_municipal_2026.xlsx", 2025)], ignore_index=True)
pemob["frota_onibus_operacional"] = pemob[["frota_onibus_convencional", "frota_onibus_articulado", "frota_onibus_biarticulado"]].sum(axis=1, min_count=1)
pemob["viagens_nao_completadas_pct"] = pemob["viagens_nao_completadas"] * 100
pemob = pemob.merge(CAPITAIS[["cod_ibge", "municipio"]])
pemob[pemob["cod_ibge"] == SALVADOR].T

,1,11
cod_ibge,2927408,2927408
frota_onibus_convencional,1789.0,1839.0
frota_onibus_articulado,NaN,1.0
frota_onibus_biarticulado,NaN,NaN
frota_micro_vans,210.0,237.0
km_onibus_ano,115000000.0,105616731.0
velocidade_brt_pico,NaN,22.2
velocidade_corredores_pico,18.0,16.6
velocidade_trafego_misto_pico,21.5,16.5
idade_media_frota,7.0,8.4


In [13]:
# Série anual do STCO: junta PEMOB + documentos oficiais, uma coluna por indicador (sem misturar definições)
stco_doc = docs[docs["tabela"].isin(["stco", "brt"])].copy()
stco_doc["coluna"] = stco_doc["tabela"] + "_" + stco_doc["indicador"]
stco_doc = stco_doc[stco_doc["ano"] <= 2025]
stco_wide = stco_doc.pivot_table(index="ano", columns="coluna", values="valor", aggfunc="first").reset_index()
stco_wide.columns.name = None
pemob_ssa = pemob[pemob["cod_ibge"] == SALVADOR][["ano", "frota_onibus_operacional", "km_onibus_ano", "velocidade_trafego_misto_pico",
                                                   "velocidade_corredores_pico", "idade_media_frota", "viagens_nao_completadas_pct",
                                                   "passageiros_equivalentes_onibus"]]
pemob_ssa = pemob_ssa.add_prefix("pemob_").rename(columns={"pemob_ano": "ano"})
stco = stco_wide.merge(pemob_ssa, on="ano", how="outer").sort_values("ano").reset_index(drop=True)
stco

,ano,brt_frota_operacional,brt_passageiros_dia_util,stco_frota_operacional,stco_idade_media_frota,stco_linhas,stco_passageiros_dia,stco_passageiros_mes,stco_viagens_mes,pemob_frota_onibus_operacional,pemob_km_onibus_ano,pemob_velocidade_trafego_misto_pico,pemob_velocidade_corredores_pico,pemob_idade_media_frota,pemob_viagens_nao_completadas_pct,pemob_passageiros_equivalentes_onibus
0,2017,NaN,NaN,NaN,NaN,530.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2018,NaN,NaN,NaN,NaN,530.0,NaN,35000000.0,410000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2019,NaN,NaN,NaN,NaN,NaN,1500000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2022,14.0,NaN,1671.0,NaN,NaN,NaN,NaN,NaN,1789.0,115000000.0,21.5,18.0,7.0,5.0,179883898.0
4,2024,NaN,NaN,NaN,NaN,NaN,1250000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2025,108.0,80000.0,1800.0,7.8,NaN,NaN,NaN,NaN,1840.0,105616731.0,16.5,16.6,8.4,16.2,162863721.0


## 7. Índice mensal de passageiros de ônibus urbanos (IMEC, SEI-BA)
A tabela do IMEC vem em PDF com o texto "espaçado" (ex.: `1 0 0 ,8 4`). Tratamento: removemos os espaços e lemos a
sequência de 36 números que vem depois do rótulo da coluna "Ônibus urbanos (2)" (jan/2023 a dez/2025).
Usamos uma única edição da tabela (dez/2025) porque a SEI revisa meses passados a cada divulgação.
Base do índice: **média de 2022 = 100**. Fonte primária dos dados: SETPS (empresas), elaboração SEI.

In [14]:
import pdfplumber

with pdfplumber.open(BRUTOS / "sei" / "TABELAS_IMEC_DEZEMBRO_2025.pdf") as pdf:
    texto = re.sub(r"\s+", "", pdf.pages[0].extract_text())


def coluna_imec(rotulo, n=36):
    inicio = texto.index(rotulo) + len(rotulo)
    return [float(v.replace(".", "").replace(",", ".")) for v in re.findall(r"\d{2,3},\d{2}", texto[inicio:])[:n]]


imec = pd.DataFrame({"data": pd.date_range("2023-01-01", "2025-12-01", freq="MS"),
                     "indice_onibus_urbanos": coluna_imec("urbanos(2)")})
imec["ano"], imec["mes"] = imec["data"].dt.year, imec["data"].dt.month
imec.groupby("ano")["indice_onibus_urbanos"].agg(["mean", "min", "max"]).round(2)

,mean,min,max
ano,,,
2023,103.03,95.89,109.36
2024,105.45,98.77,112.90
2025,106.66,101.84,112.07


## 8. Deslocamento casa-trabalho (Censo 2010 e 2022)
* **Meio de transporte (2022, tab. 10332):** percentual de trabalhadores por meio em que passam mais tempo;
  agrupamos em *transporte coletivo* (ônibus + BRT + trem/metrô + van), *individual motorizado*
  (automóvel + moto + mototáxi + táxi) e *ativo* (a pé + bicicleta).
* **Tempo (2010, tab. 3604 × 2022, tab. 10330):** as faixas do Censo 2022 são mais detalhadas; harmonizamos em
  quatro faixas comuns (até 30 min, 30 min–1 h, 1–2 h, mais de 2 h). Os universos são próximos
  (2010: retornam diariamente; 2022: retornam 3 ou mais dias por semana).
* **Tempo médio por meio (2022, tab. 10330):** estimado pelo ponto médio de cada faixa
  (3; 10,5; 22,5; 45; 90; 180; 270 minutos). É uma **aproximação**, documentada como tal.

In [15]:
modo = ler_sidra(BRUTOS / "ibge" / "sidra_10332_censo2022_deslocamento.json")
modo = modo[modo["Unidade de Medida"] == "Pessoas"]
col_modo = "Meio de transporte em que passa mais tempo para chegar ao local de trabalho"
modo_w = modo.pivot_table(index="cod_ibge", columns=col_modo, values="Valor", aggfunc="sum")
GRUPOS = {
    "pct_onibus": ["Ônibus"], "pct_brt": ["BRT ou ônibus de trânsito rápido"], "pct_trem_metro": ["Trem ou metrô"],
    "pct_automovel": ["Automóvel"], "pct_moto": ["Motocicleta", "Mototáxi"], "pct_a_pe": ["A pé"],
    "pct_bicicleta": ["Bicicleta"],
    "pct_coletivo": ["Ônibus", "BRT ou ônibus de trânsito rápido", "Trem ou metrô", "Van, perua ou assemelhados"],
    "pct_individual_motorizado": ["Automóvel", "Motocicleta", "Mototáxi", "Táxi ou assemelhados"],
    "pct_ativo": ["A pé", "Bicicleta"],
}
modal = pd.DataFrame({k: modo_w[v].sum(axis=1) / modo_w["Total"] * 100 for k, v in GRUPOS.items()}).reset_index()
modal["trabalhadores_que_se_deslocam"] = modo_w["Total"].values

tempo22 = ler_sidra(BRUTOS / "ibge" / "sidra_10330_censo2022_deslocamento.json")
tempo22 = tempo22[tempo22["Unidade de Medida"] == "Pessoas"]
col_t = "Tempo habitual de deslocamento do domicílio para o trabalho principal"
MEIO_MIN = {"Até cinco minutos": 3, "De seis minutos até quinze minutos": 10.5, "Mais de quinze minutos até meia hora": 22.5,
            "Mais de meia hora até uma hora": 45, "Mais de uma hora até duas horas": 90,
            "Mais de duas horas até quatro horas": 180, "Mais de quatro horas": 270}
FAIXA4_22 = {"Até cinco minutos": "ate_30min", "De seis minutos até quinze minutos": "ate_30min",
             "Mais de quinze minutos até meia hora": "ate_30min", "Mais de meia hora até uma hora": "30min_1h",
             "Mais de uma hora até duas horas": "1h_2h", "Mais de duas horas até quatro horas": "mais_2h",
             "Mais de quatro horas": "mais_2h"}
t22 = tempo22[(tempo22[col_modo] == "Total") & (tempo22[col_t] != "Total")].copy()
t22["faixa"] = t22[col_t].map(FAIXA4_22)
dist22 = t22.pivot_table(index="cod_ibge", columns="faixa", values="Valor", aggfunc="sum")
dist22 = dist22.div(dist22.sum(axis=1), axis=0).mul(100).add_prefix("pct_2022_").reset_index()

tempo10 = ler_sidra(BRUTOS / "ibge" / "sidra_3604_censo2010_deslocamento.json")
tempo10 = tempo10[(tempo10["Unidade de Medida"] == "Pessoas") & (tempo10[col_t] != "Total")].copy()
FAIXA4_10 = {"Até cinco minutos": "ate_30min", "De seis minutos até meia hora": "ate_30min",
             "Mais de meia hora até uma hora": "30min_1h", "Mais de uma hora até duas horas": "1h_2h", "Mais de duas horas": "mais_2h"}
tempo10["faixa"] = tempo10[col_t].map(FAIXA4_10)
dist10 = tempo10.pivot_table(index="cod_ibge", columns="faixa", values="Valor", aggfunc="sum")
dist10 = dist10.div(dist10.sum(axis=1), axis=0).mul(100).add_prefix("pct_2010_").reset_index()

# tempo médio estimado (minutos) por meio de transporte, 2022
tm = tempo22[(tempo22[col_t] != "Total") & (tempo22[col_modo] != "Total")].copy()
tm["minutos"] = tm[col_t].map(MEIO_MIN)
tempo_medio_modo = (tm.assign(peso=tm["Valor"] * tm["minutos"]).groupby(["cod_ibge", col_modo])[["peso", "Valor"]].sum()
                    .assign(tempo_medio_min=lambda x: x["peso"] / x["Valor"]).reset_index()
                    .rename(columns={col_modo: "meio_transporte", "Valor": "trabalhadores"})[["cod_ibge", "meio_transporte", "trabalhadores", "tempo_medio_min"]])
tempo_geral = (tm.assign(peso=tm["Valor"] * tm["minutos"]).groupby("cod_ibge")[["peso", "Valor"]].sum()
               .assign(tempo_medio_2022_min=lambda x: x["peso"] / x["Valor"])[["tempo_medio_2022_min"]].reset_index())

deslocamento = CAPITAIS[["cod_ibge", "municipio"]].merge(modal).merge(dist10).merge(dist22).merge(tempo_geral)
deslocamento["pct_mais_1h_2010"] = deslocamento["pct_2010_1h_2h"] + deslocamento["pct_2010_mais_2h"]
deslocamento["pct_mais_1h_2022"] = deslocamento["pct_2022_1h_2h"] + deslocamento["pct_2022_mais_2h"]
deslocamento.round(1)

,cod_ibge,municipio,pct_onibus,pct_brt,pct_trem_metro,pct_automovel,pct_moto,pct_a_pe,pct_bicicleta,pct_coletivo,pct_individual_motorizado,pct_ativo,trabalhadores_que_se_deslocam,pct_2010_1h_2h,pct_2010_30min_1h,pct_2010_ate_30min,pct_2010_mais_2h,pct_2022_1h_2h,pct_2022_30min_1h,pct_2022_ate_30min,pct_2022_mais_2h,tempo_medio_2022_min,pct_mais_1h_2010,pct_mais_1h_2022
0,2927408,Salvador,44.7,0.7,2.2,26.0,4.2,15.1,1.3,48.4,31.3,16.4,864427.0,19.5,38.6,39.3,2.6,20.4,34.9,43.5,1.3,43.8,22.0,21.7
1,3550308,São Paulo,36.8,0.4,11.1,29.5,4.3,12.0,1.0,48.9,34.7,13.0,4251307.0,25.2,35.4,33.6,5.9,26.1,32.9,37.7,3.3,50.6,31.0,29.4
2,3304557,Rio de Janeiro,34.6,4.2,8.6,24.3,3.8,12.9,2.8,50.1,29.9,15.7,2146204.0,21.3,36.3,38.4,4.0,23.8,32.3,39.9,3.9,49.8,25.3,27.8
3,5300108,Brasília,32.1,1.6,2.8,45.2,3.9,8.3,2.4,36.8,49.6,10.7,1157926.0,15.4,36.2,47.3,1.1,14.5,32.2,52.7,0.7,37.4,16.5,15.1
4,2304400,Fortaleza,29.2,0.3,0.6,28.5,15.0,13.8,7.1,30.6,44.7,21.0,857277.0,11.6,33.6,53.7,1.1,14.6,29.0,55.6,0.8,36.6,12.7,15.4
5,3106200,Belo Horizonte,37.0,1.0,0.8,34.6,6.6,13.9,0.7,39.2,42.6,14.5,938713.0,15.3,37.4,46.1,1.2,16.7,32.4,49.8,1.1,39.8,16.6,17.8
6,1302603,Manaus,40.0,0.5,0.0,28.5,9.8,11.1,0.6,43.2,39.6,11.7,660061.0,15.1,39.0,43.6,2.2,19.9,33.0,45.7,1.4,42.8,17.3,21.2
7,4106902,Curitiba,26.4,0.4,0.0,49.5,5.0,10.9,2.7,27.7,55.4,13.6,739864.0,10.0,33.4,55.7,0.9,10.1,27.5,62.0,0.4,32.2,10.8,10.5
8,2611606,Recife,32.0,0.8,1.0,32.0,7.7,14.2,6.9,34.1,41.2,21.1,535041.0,10.2,34.8,53.9,1.1,14.3,30.9,53.9,0.9,37.4,11.2,15.2
9,5208707,Goiânia,16.3,0.2,0.0,48.1,15.0,11.2,4.3,16.7,64.4,15.4,595455.0,8.1,26.2,64.5,1.2,9.1,23.0,67.4,0.5,30.3,9.3,9.6


In [16]:
tempo_medio_modo[tempo_medio_modo["cod_ibge"] == SALVADOR].sort_values("trabalhadores", ascending=False).round(1)

,cod_ibge,meio_transporte,trabalhadores,tempo_medio_min
83,2927408,Ônibus,364185.0,59.2
71,2927408,Automóvel,212342.0,32.5
70,2927408,A pé,123804.0,21.0
77,2927408,Motocicleta,30072.0,31.0
80,2927408,Trem ou metrô,18131.0,49.1
73,2927408,Bicicleta,10321.0,32.7
81,2927408,Táxi ou assemelhados,9313.0,29.6
82,2927408,"Van, perua ou assemelhados",6687.0,53.0
72,2927408,BRT ou ônibus de trânsito rápido,6016.0,63.2
78,2927408,Mototáxi,4579.0,29.2


## 9. Integração e criação de indicadores
### 9.1 Painel das capitais (capital × ano, 2014–2025)
Une população (IBGE) e frota (SENATRAN) e cria indicadores por habitante:
* `autos_por_mil_hab`, `motos_por_mil_hab`, `motorizacao_por_mil_hab` (autos + motos por 1.000 hab.);
* `onibus_reg_por_100mil_hab`;
* variações em relação a 2014 (`var_pop_desde_2014_pct`, `var_motorizacao_desde_2014_pct`).

In [17]:
painel = (frota.merge(populacao[["cod_ibge", "ano", "populacao"]], on=["cod_ibge", "ano"])
          .merge(CAPITAIS[["cod_ibge", "municipio", "uf", "regiao"]]).merge(territorio))
painel["autos_por_mil_hab"] = painel["autos"] / painel["populacao"] * 1000
painel["motos_por_mil_hab"] = painel["motos"] / painel["populacao"] * 1000
painel["motorizacao_por_mil_hab"] = (painel["autos"] + painel["motos"]) / painel["populacao"] * 1000
painel["onibus_reg_por_100mil_hab"] = painel["onibus_registrados"] / painel["populacao"] * 100_000
base14 = painel[painel["ano"] == 2014].set_index("cod_ibge")
painel["var_pop_desde_2014_pct"] = (painel["populacao"] / painel["cod_ibge"].map(base14["populacao"]) - 1) * 100
painel["var_motorizacao_desde_2014_pct"] = (painel["motorizacao_por_mil_hab"] / painel["cod_ibge"].map(base14["motorizacao_por_mil_hab"]) - 1) * 100
painel["var_frota_autos_motos_desde_2014_pct"] = ((painel["autos"] + painel["motos"]) /
                                                   painel["cod_ibge"].map(base14["autos"] + base14["motos"]) - 1) * 100
cols_painel = ["cod_ibge", "municipio", "uf", "regiao", "ano", "populacao", "densidade_2022", "frota_total", "automovel", "autos",
               "motos", "onibus_registrados", "autos_por_mil_hab", "motos_por_mil_hab", "motorizacao_por_mil_hab",
               "onibus_reg_por_100mil_hab", "var_pop_desde_2014_pct", "var_motorizacao_desde_2014_pct",
               "var_frota_autos_motos_desde_2014_pct"]
painel = painel[cols_painel].sort_values(["municipio", "ano"]).reset_index(drop=True)
print(painel.shape)
painel[painel["cod_ibge"] == SALVADOR].round(1)

(156, 19)


,cod_ibge,municipio,uf,regiao,ano,populacao,densidade_2022,frota_total,automovel,autos,motos,onibus_registrados,autos_por_mil_hab,motos_por_mil_hab,motorizacao_por_mil_hab,onibus_reg_por_100mil_hab,var_pop_desde_2014_pct,var_motorizacao_desde_2014_pct,var_frota_autos_motos_desde_2014_pct
120,2927408,Salvador,BA,Nordeste,2014,2586741,3486.5,819237,551687,655553,119584,13016,253.4,46.2,299.7,503.2,0.0,0.0,0.0
121,2927408,Salvador,BA,Nordeste,2015,2564978,3486.5,846102,564606,674223,126513,13638,262.9,49.3,312.2,531.7,-0.8,4.2,3.3
122,2927408,Salvador,BA,Nordeste,2016,2543398,3486.5,866032,573138,688002,132216,13478,270.5,52.0,322.5,529.9,-1.7,7.6,5.8
123,2927408,Salvador,BA,Nordeste,2017,2522000,3486.5,886703,583805,703215,137745,13126,278.8,54.6,333.4,520.5,-2.5,11.3,8.5
124,2927408,Salvador,BA,Nordeste,2018,2500781,3486.5,915186,597610,723830,145179,12796,289.4,58.1,347.5,511.7,-3.3,16.0,12.1
125,2927408,Salvador,BA,Nordeste,2019,2479742,3486.5,945305,610461,743710,153767,13160,299.9,62.0,361.9,530.7,-4.1,20.8,15.8
126,2927408,Salvador,BA,Nordeste,2020,2458879,3486.5,964263,615701,753288,161491,13556,306.4,65.7,372.0,551.3,-4.9,24.2,18.0
127,2927408,Salvador,BA,Nordeste,2021,2438191,3486.5,982619,619056,760993,170329,14062,312.1,69.9,382.0,576.7,-5.7,27.5,20.1
128,2927408,Salvador,BA,Nordeste,2022,2417678,3486.5,1006292,623395,771509,181511,14512,319.1,75.1,394.2,600.2,-6.5,31.5,22.9
129,2927408,Salvador,BA,Nordeste,2023,2413232,3486.5,1036885,630657,786187,194844,14953,325.8,80.7,406.5,619.6,-6.7,35.7,26.6


### 9.2 PEMOB das capitais (indicadores de oferta por habitante)
Para comparar Salvador com as outras capitais, a frota operacional e os passageiros da PEMOB são divididos pela
população compatibilizada do mesmo ano.

In [18]:
pemob_cap = pemob.merge(populacao[["cod_ibge", "ano", "populacao"]], on=["cod_ibge", "ano"], how="left")
pemob_cap["onibus_operacionais_por_100mil_hab"] = pemob_cap["frota_onibus_operacional"] / pemob_cap["populacao"] * 100_000
pemob_cap["pass_equiv_por_hab_ano"] = pemob_cap["passageiros_equivalentes_onibus"] / pemob_cap["populacao"]
pemob_cap["km_onibus_por_hab_ano"] = pemob_cap["km_onibus_ano"] / pemob_cap["populacao"]
pemob_cap["pass_equiv_por_onibus_dia"] = pemob_cap["passageiros_equivalentes_onibus"] / pemob_cap["frota_onibus_operacional"] / 365
pemob_cap = pemob_cap.drop(columns=["viagens_nao_completadas"])
pemob_cap.round(2)

,cod_ibge,frota_onibus_convencional,frota_onibus_articulado,frota_onibus_biarticulado,frota_micro_vans,km_onibus_ano,velocidade_brt_pico,velocidade_corredores_pico,velocidade_trafego_misto_pico,idade_media_frota,tarifa_pemob,receita_tarifaria_onibus,passageiros_equivalentes_onibus,ano,edicao_pemob,frota_onibus_operacional,viagens_nao_completadas_pct,municipio,populacao,onibus_operacionais_por_100mil_hab,pass_equiv_por_hab_ano,km_onibus_por_hab_ano,pass_equiv_por_onibus_dia
0,1302603,NaN,NaN,NaN,NaN,NaN,NaN,NaN,25.0,8.8,4.50,NaN,NaN,2022,2023,NaN,5.0,Manaus,2063689,NaN,NaN,NaN,NaN
1,2927408,1789.0,NaN,NaN,210.0,115000000.0,NaN,18.0,21.5,7.0,4.90,8.816118e+08,179883898.0,2022,2023,1789.0,5.0,Salvador,2417678,74.00,74.40,47.57,275.48
2,5208707,1113.0,135.0,28.0,NaN,NaN,NaN,NaN,NaN,NaN,4.30,NaN,NaN,2022,2023,1276.0,1.1,Goiânia,1437366,88.77,NaN,NaN,NaN
3,3106200,2212.0,189.0,NaN,34.0,118186626.0,44.5,14.0,9.0,NaN,4.50,8.625001e+08,191666698.0,2022,2023,2401.0,0.0,Belo Horizonte,2315560,103.69,82.77,51.04,218.71
4,1501402,649.0,15.0,NaN,247.0,675464234.0,26.5,17.4,17.5,8.3,4.00,NaN,12646277.0,2022,2023,664.0,NaN,Belém,1303403,50.94,9.70,518.23,52.18
5,2611606,NaN,NaN,NaN,154.0,4173192.0,25.0,19.0,13.0,5.4,4.10,9.713559e+08,223112593.0,2022,2023,NaN,NaN,Recife,1488920,NaN,149.85,2.80,NaN
6,4106902,727.0,227.0,138.0,NaN,NaN,22.8,NaN,17.3,NaN,6.00,NaN,NaN,2022,2023,1092.0,1.0,Curitiba,1773718,61.57,NaN,NaN,NaN
7,3304557,3600.0,561.0,NaN,NaN,NaN,NaN,NaN,16.2,NaN,4.30,NaN,NaN,2022,2023,4161.0,0.0,Rio de Janeiro,6211223,66.99,NaN,NaN,NaN
8,4314902,966.0,133.0,NaN,404.0,60385075.0,NaN,20.9,21.2,7.6,4.80,NaN,117554071.0,2022,2023,1099.0,0.8,Porto Alegre,1332845,82.46,88.20,45.31,293.05
9,3550308,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.40,NaN,NaN,2022,2023,NaN,42.0,São Paulo,11451999,NaN,NaN,NaN,NaN


### 9.3 Base analítica anual de Salvador (2010–2025)
Uma linha por ano. Junta população, frota, tarifa, metrô e STCO e cria variáveis analíticas:
* `metro_passageiros_por_hab`: viagens de metrô por habitante no ano;
* `stco_passageiros_dia_semob` (2019 e 2024) e `pemob_passageiros_equivalentes_onibus` (2022 e 2025);
* `stco_onibus_por_100mil_hab`: frota operacional da PEMOB por 100 mil habitantes;
* `hab_por_linha`: habitantes por linha de ônibus (anos com nº de linhas oficial);
* marcadores de contexto: `pandemia` (2020–2021), `metro_em_operacao` (2014+), `brt_em_operacao` (2022+) e
  `fase` (antes do metrô / metrô em expansão / pandemia / pós-pandemia).

In [19]:
anos = pd.DataFrame({"ano": range(2010, 2026)})
ssa_pop = populacao[populacao["cod_ibge"] == SALVADOR][["ano", "populacao", "pop_estimativa_oficial", "metodo_pop"]]
ssa_frota = painel[painel["cod_ibge"] == SALVADOR][["ano", "autos", "motos", "onibus_registrados", "autos_por_mil_hab",
                                                    "motos_por_mil_hab", "motorizacao_por_mil_hab"]]
base = (anos.merge(ssa_pop, how="left").merge(ssa_frota, how="left")
        .merge(tarifa[["ano", "tarifa_media_ano", "tarifa_dezembro", "tarifa_real_dez2025"]], how="left")
        .merge(metro_anual[["ano", "metro_passageiros_ano", "metro_precisao"]], how="left")
        .merge(stco.rename(columns={"stco_passageiros_dia": "stco_passageiros_dia_semob", "stco_linhas": "stco_linhas"}), how="left"))
base["metro_passageiros_por_hab"] = base["metro_passageiros_ano"] / base["populacao"]
base["stco_onibus_por_100mil_hab"] = base["pemob_frota_onibus_operacional"] / base["populacao"] * 100_000
base["hab_por_linha"] = base["populacao"] / base["stco_linhas"]
base["pandemia"] = base["ano"].isin([2020, 2021]).astype(int)
base["metro_em_operacao"] = (base["ano"] >= 2014).astype(int)
base["brt_em_operacao"] = (base["ano"] >= 2022).astype(int)
base["fase"] = np.select([base["ano"] < 2014, base["ano"] < 2020, base["ano"] < 2022],
                         ["antes do metrô", "metrô em expansão", "pandemia"], "pós-pandemia")
print(base.shape)
base.round(2)

(16, 37)


,ano,populacao,pop_estimativa_oficial,metodo_pop,autos,motos,onibus_registrados,autos_por_mil_hab,motos_por_mil_hab,motorizacao_por_mil_hab,tarifa_media_ano,tarifa_dezembro,tarifa_real_dez2025,metro_passageiros_ano,metro_precisao,brt_frota_operacional,brt_passageiros_dia_util,stco_frota_operacional,stco_idade_media_frota,stco_linhas,stco_passageiros_dia_semob,stco_passageiros_mes,stco_viagens_mes,pemob_frota_onibus_operacional,pemob_km_onibus_ano,pemob_velocidade_trafego_misto_pico,pemob_velocidade_corredores_pico,pemob_idade_media_frota,pemob_viagens_nao_completadas_pct,pemob_passageiros_equivalentes_onibus,metro_passageiros_por_hab,stco_onibus_por_100mil_hab,hab_por_linha,pandemia,metro_em_operacao,brt_em_operacao,fase
0,2010,2675656,NaN,censo,NaN,NaN,NaN,NaN,NaN,NaN,2.30,2.3,5.46,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,antes do metrô
1,2011,2653145,2693606.0,interpolacao_intercensitaria,NaN,NaN,NaN,NaN,NaN,NaN,2.50,2.5,5.57,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,antes do metrô
2,2012,2630823,2710968.0,interpolacao_intercensitaria,NaN,NaN,NaN,NaN,NaN,NaN,2.67,2.8,5.65,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,antes do metrô
3,2013,2608689,2883682.0,interpolacao_intercensitaria,NaN,NaN,NaN,NaN,NaN,NaN,2.80,2.8,5.58,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,antes do metrô
4,2014,2586741,2902927.0,interpolacao_intercensitaria,655553.0,119584.0,13016.0,253.43,46.23,299.66,2.80,2.8,5.24,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1,0,metrô em expansão
5,2015,2564978,2921087.0,interpolacao_intercensitaria,674223.0,126513.0,13638.0,262.86,49.32,312.18,3.00,3.0,5.15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1,0,metrô em expansão
6,2016,2543398,2938092.0,interpolacao_intercensitaria,688002.0,132216.0,13478.0,270.51,51.98,322.49,3.30,3.3,5.21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1,0,metrô em expansão
7,2017,2522000,2953986.0,interpolacao_intercensitaria,703215.0,137745.0,13126.0,278.83,54.62,333.45,3.60,3.6,5.50,NaN,NaN,NaN,NaN,NaN,NaN,530.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4758.49,0,1,0,metrô em expansão
8,2018,2500781,2857329.0,interpolacao_intercensitaria,723830.0,145179.0,12796.0,289.44,58.05,347.50,3.70,3.7,5.45,90000000.0,aproximado (quase 90 milhoes),NaN,NaN,NaN,NaN,530.0,NaN,35000000.0,410000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,35.99,NaN,4718.45,0,1,0,metrô em expansão
9,2019,2479742,2872347.0,interpolacao_intercensitaria,743710.0,153767.0,13160.0,299.91,62.01,361.92,4.00,4.0,5.68,107431208.0,exato (corrigido na transformação),NaN,NaN,NaN,NaN,NaN,1500000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,43.32,NaN,NaN,0,1,0,metrô em expansão


### 9.4 Base mensal de Salvador (jan/2023–dez/2025)
Junta o índice de passageiros de ônibus (IMEC), os passageiros do metrô (CTB, a partir de 2024) e a tarifa vigente.
O metrô também é convertido em índice (média de 2024 = 100) para comparar tendências na mesma escala.

In [20]:
mensal = imec.merge(metro_mensal[["data", "passageiros"]].rename(columns={"passageiros": "metro_passageiros_mes"}), on="data", how="left")
tarifa_mes = dias.assign(data=dias["data"].dt.to_period("M").dt.to_timestamp()).groupby("data")["tarifa"].mean().rename("tarifa_vigente")
mensal = mensal.merge(tarifa_mes.reset_index(), on="data", how="left")
media_metro_24 = mensal.loc[mensal["ano"] == 2024, "metro_passageiros_mes"].mean()
mensal["indice_metro_2024_100"] = mensal["metro_passageiros_mes"] / media_metro_24 * 100
media_onibus_24 = mensal.loc[mensal["ano"] == 2024, "indice_onibus_urbanos"].mean()
mensal["indice_onibus_2024_100"] = mensal["indice_onibus_urbanos"] / media_onibus_24 * 100
mensal = mensal[["data", "ano", "mes", "indice_onibus_urbanos", "indice_onibus_2024_100", "metro_passageiros_mes",
                 "indice_metro_2024_100", "tarifa_vigente"]]
mensal.tail(6).round(2)

,data,ano,mes,indice_onibus_urbanos,indice_onibus_2024_100,metro_passageiros_mes,indice_metro_2024_100,tarifa_vigente
30,2025-07-01,2025,7,108.02,102.44,9979955.0,101.87,5.6
31,2025-08-01,2025,8,101.87,96.61,10053569.0,102.62,5.6
32,2025-09-01,2025,9,108.01,102.43,10414494.0,106.31,5.6
33,2025-10-01,2025,10,107.63,102.07,10514842.0,107.33,5.6
34,2025-11-01,2025,11,104.27,98.88,9703140.0,99.05,5.6
35,2025-12-01,2025,12,104.11,98.73,9400344.0,95.96,5.6


## 10. Carga (SQLite + CSV) e validação final
Todas as tabelas são gravadas no banco **SQLite** `data/db/mobilidade_salvador.sqlite` (pode ser aberto com
DB Browser for SQLite ou consultado com SQL) e exportadas em CSV (UTF-8) para o dashboard.
Também é gerado um **dicionário de dados** com a descrição e a fonte de cada coluna.

In [21]:
TABELAS = {
    "base_salvador_anual": base,
    "base_salvador_mensal": mensal,
    "painel_capitais_anual": painel,
    "pemob_capitais": pemob_cap,
    "deslocamento_capitais_censo": deslocamento,
    "tempo_medio_por_meio_censo2022": tempo_medio_modo.merge(CAPITAIS[["cod_ibge", "municipio"]]),
    "populacao_capitais": populacao.merge(CAPITAIS[["cod_ibge", "municipio"]]),
    "frota_capitais": frota.merge(CAPITAIS[["cod_ibge", "municipio"]]),
    "tarifa_anual": tarifa,
    "metro_mensal": metro_mensal,
    "stco_salvador": stco,
    "fontes_documentais": docs,
}
if BANCO.exists():
    BANCO.unlink()
with sqlite3.connect(BANCO) as con:
    for nome, df in TABELAS.items():
        df2 = df.copy()
        for c in df2.columns:
            if pd.api.types.is_datetime64_any_dtype(df2[c]):
                df2[c] = df2[c].dt.strftime("%Y-%m-%d")
        df2.to_sql(nome, con, index=False)
        df.to_csv(PROCESSADOS / f"{nome}.csv", index=False, encoding="utf-8")
    resumo = pd.read_sql("SELECT name AS tabela FROM sqlite_master WHERE type='table'", con)
    resumo["linhas"] = [pd.read_sql(f"SELECT COUNT(*) n FROM {t}", con)["n"][0] for t in resumo["tabela"]]
    resumo["colunas"] = [len(pd.read_sql(f"SELECT * FROM {t} LIMIT 1", con).columns) for t in resumo["tabela"]]
resumo

,tabela,linhas,colunas
0,base_salvador_anual,16,37
1,base_salvador_mensal,36,8
2,painel_capitais_anual,156,19
3,pemob_capitais,21,23
4,deslocamento_capitais_censo,13,24
5,tempo_medio_por_meio_censo2022,182,5
6,populacao_capitais,221,6
7,frota_capitais,156,8
8,tarifa_anual,16,5
9,metro_mensal,33,4


In [22]:
# Exemplo de consulta SQL no banco carregado: motorização das capitais em 2025
with sqlite3.connect(BANCO) as con:
    consulta = pd.read_sql("""
        SELECT municipio, populacao, ROUND(motorizacao_por_mil_hab, 1) AS motorizacao_por_mil_hab,
               ROUND(var_motorizacao_desde_2014_pct, 1) AS var_desde_2014_pct
        FROM painel_capitais_anual WHERE ano = 2025 ORDER BY motorizacao_por_mil_hab DESC""", con)
consulta

,municipio,populacao,motorizacao_por_mil_hab,var_desde_2014_pct
0,Belo Horizonte,2314218,1176.1,74.7
1,Curitiba,1778289,1045.6,32.0
2,Goiânia,1462487,872.1,18.8
3,São Paulo,11479120,829.1,33.7
4,Brasília,2857470,747.6,30.5
5,Porto Alegre,1331326,666.5,17.7
6,Rio de Janeiro,6213535,511.7,29.8
7,Fortaleza,2440248,510.5,38.0
8,Recife,1490803,487.7,25.0
9,São Luís,1041092,463.6,44.7


### 10.1 Validações automáticas (testes de qualidade)
Cada teste deve resultar em `True`. Se algum falhar, o notebook para (`assert`).

In [23]:
testes = {
    # cada subpasta de data/raw é um órgão diferente (ibge, senatran, pemob, sei, ctb, documentos)
    "pelo menos 3 fontes oficiais distintas": len({p.relative_to(BRUTOS).parts[0] for p in BRUTOS.rglob("*")
                                                    if p.is_file() and len(p.relative_to(BRUTOS).parts) > 1}) >= 3,
    "base anual de Salvador com >= 6 colunas": base.shape[1] >= 6,
    "painel sem duplicidade capital-ano": not painel.duplicated(["cod_ibge", "ano"]).any(),
    "13 capitais x 12 anos no painel": painel.shape[0] == 13 * 12,
    "frota: soma dos municípios = total nacional (todos os anos)": all(v["confere"] for v in validacoes),
    "SHA-256 dos arquivos baixados pelo navegador confere": all(c["sha256_confere"] for c in checagem),
    "população de Salvador no Censo 2022 = 2.417.678": int(censos.set_index("cod_ibge").loc[SALVADOR, "censo_2022"]) == 2417678,
    "metrô mensal 2024 difere < 0,1% do total oficial": abs(validacao_metro.loc[2024, "soma_meses"] / 117533257 - 1) < 0.001,
    "IMEC: 36 meses lidos": len(imec) == 36,
    "percentuais modais somam no máximo 100% (coletivo + individual + ativo)":
        bool((deslocamento[["pct_coletivo", "pct_individual_motorizado", "pct_ativo"]].sum(axis=1) <= 100.01).all()),
    "sem valores negativos nas contagens": bool((painel[["populacao", "autos", "motos"]] >= 0).all().all()),
}
for nome, ok in testes.items():
    print(("OK   " if ok else "FALHA"), nome)
# registros de validação gravados para documentação (usados no relatório de ETL)
pd.DataFrame([{"teste": k, "resultado": "OK" if v else "FALHA"} for k, v in testes.items()]).to_csv(
    PROCESSADOS / "testes_qualidade.csv", index=False, encoding="utf-8")
pd.DataFrame(validacoes).to_csv(PROCESSADOS / "validacao_frota_senatran.csv", index=False, encoding="utf-8")
pd.DataFrame(checagem).to_csv(PROCESSADOS / "validacao_sha256_senatran.csv", index=False, encoding="utf-8")
assert all(testes.values()), "Algum teste de qualidade falhou"

OK    pelo menos 3 fontes oficiais distintas
OK    base anual de Salvador com >= 6 colunas
OK    painel sem duplicidade capital-ano
OK    13 capitais x 12 anos no painel
OK    frota: soma dos municípios = total nacional (todos os anos)
OK    SHA-256 dos arquivos baixados pelo navegador confere
OK    população de Salvador no Censo 2022 = 2.417.678
OK    metrô mensal 2024 difere < 0,1% do total oficial
OK    IMEC: 36 meses lidos
OK    percentuais modais somam no máximo 100% (coletivo + individual + ativo)
OK    sem valores negativos nas contagens


In [24]:
# Dicionário de dados (descrição de cada coluna das tabelas principais)
DESCRICOES = {
    "ano": "Ano de referência", "populacao": "População residente compatibilizada (IBGE; ver seção 2)",
    "pop_estimativa_oficial": "Estimativa oficial do IBGE publicada para o ano (sem compatibilização)",
    "metodo_pop": "Como a população do ano foi obtida", "autos": "Veículos leves emplacados (SENATRAN, dezembro)",
    "motos": "Motocicletas, motonetas e ciclomotores emplacados (SENATRAN, dezembro)",
    "onibus_registrados": "Ônibus + micro-ônibus emplacados no município (SENATRAN), inclui fretamento",
    "autos_por_mil_hab": "Autos por 1.000 habitantes", "motos_por_mil_hab": "Motos por 1.000 habitantes",
    "motorizacao_por_mil_hab": "(Autos + motos) por 1.000 habitantes",
    "tarifa_media_ano": "Tarifa do ônibus ponderada pelos dias de vigência no ano (R$ correntes)",
    "tarifa_dezembro": "Tarifa vigente em 31/dez (R$)", "tarifa_real_dez2025": "Tarifa média do ano em R$ de dez/2025 (IPCA)",
    "metro_passageiros_ano": "Passageiros do metrô no ano (Cia. do Metrô da Bahia / CTB)",
    "metro_precisao": "Precisão declarada pela fonte (exato ou aproximado)",
    "metro_passageiros_por_hab": "Viagens de metrô por habitante no ano",
    "stco_passageiros_dia_semob": "Passageiros por dia do STCO (SEMOB)",
    "stco_linhas": "Número de linhas do STCO (Prefeitura)",
    "pemob_frota_onibus_operacional": "Frota operacional de ônibus do sistema municipal (PEMOB)",
    "pemob_km_onibus_ano": "Quilômetros rodados pela frota de ônibus no ano (PEMOB)",
    "pemob_velocidade_trafego_misto_pico": "Velocidade média dos ônibus em tráfego misto no pico, km/h (PEMOB)",
    "pemob_velocidade_corredores_pico": "Velocidade média nos corredores de ônibus no pico, km/h (PEMOB)",
    "pemob_idade_media_frota": "Idade média da frota de ônibus, anos (PEMOB)",
    "pemob_viagens_nao_completadas_pct": "Viagens de ônibus programadas e não completadas, % (PEMOB)",
    "pemob_passageiros_equivalentes_onibus": "Passageiros equivalentes no ano = receita ÷ tarifa (PEMOB)",
    "stco_onibus_por_100mil_hab": "Ônibus operacionais (PEMOB) por 100 mil habitantes",
    "hab_por_linha": "Habitantes por linha de ônibus", "pandemia": "1 = anos de pandemia (2020-2021)",
    "metro_em_operacao": "1 = metrô em operação", "brt_em_operacao": "1 = BRT em operação", "fase": "Período histórico",
    "indice_onibus_urbanos": "Índice de passageiros de ônibus urbanos, média 2022 = 100 (SEI-BA/IMEC)",
    "indice_onibus_2024_100": "Mesmo índice rebaseado para média de 2024 = 100",
    "metro_passageiros_mes": "Passageiros do metrô no mês (CTB)", "indice_metro_2024_100": "Metrô em índice, média 2024 = 100",
    "tarifa_vigente": "Tarifa média vigente no mês (R$)",
    # identificação
    "cod_ibge": "Código IBGE do município (7 dígitos)", "municipio": "Nome do município", "uf": "Sigla da UF",
    "regiao": "Grande região (IBGE)", "data": "Primeiro dia do mês de referência", "mes": "Mês de referência (1 a 12)",
    # documentos oficiais do STCO/BRT
    "brt_frota_operacional": "Ônibus em operação no BRT (Prefeitura)",
    "brt_passageiros_dia_util": "Passageiros do BRT por dia útil (SEMOB)",
    "stco_frota_operacional": "Frota operacional do STCO segundo a Prefeitura (relatórios e anúncios oficiais)",
    "stco_idade_media_frota": "Idade média da frota do STCO, anos (SEMOB)",
    "stco_passageiros_mes": "Passageiros do STCO por mês (Relatório de Atividades da Prefeitura)",
    "stco_viagens_mes": "Viagens de ônibus realizadas por mês (Relatório de Atividades da Prefeitura)",
    # painel das capitais
    "densidade_2022": "Habitantes por km² (Censo 2022)", "frota_total": "Todos os veículos emplacados (SENATRAN)",
    "automovel": "Somente automóveis (SENATRAN)", "onibus_reg_por_100mil_hab": "Ônibus emplacados por 100 mil hab.",
    "var_pop_desde_2014_pct": "Variação da população desde 2014, %",
    "var_motorizacao_desde_2014_pct": "Variação da motorização desde 2014, %",
    "var_frota_autos_motos_desde_2014_pct": "Variação do número de autos + motos desde 2014, %",
    # PEMOB das capitais
    "frota_onibus_convencional": "Ônibus convencionais do sistema municipal (PEMOB)",
    "frota_onibus_articulado": "Ônibus articulados (PEMOB)", "frota_onibus_biarticulado": "Ônibus biarticulados (PEMOB)",
    "frota_micro_vans": "Micro-ônibus e vans do sistema municipal (PEMOB)",
    "km_onibus_ano": "Km rodados pelos ônibus no ano (PEMOB)", "velocidade_brt_pico": "Velocidade do BRT no pico, km/h (PEMOB)",
    "velocidade_corredores_pico": "Velocidade nos corredores no pico, km/h (PEMOB)",
    "velocidade_trafego_misto_pico": "Velocidade em tráfego misto no pico, km/h (PEMOB)",
    "idade_media_frota": "Idade média da frota, anos (PEMOB)", "tarifa_pemob": "Tarifa declarada na PEMOB (R$)",
    "receita_tarifaria_onibus": "Receita tarifária anual dos ônibus, R$ (PEMOB)",
    "passageiros_equivalentes_onibus": "Passageiros equivalentes no ano = receita ÷ tarifa (PEMOB)",
    "edicao_pemob": "Edição da PEMOB (ano de publicação)",
    "frota_onibus_operacional": "Ônibus convencionais + articulados + biarticulados (PEMOB)",
    "viagens_nao_completadas_pct": "Viagens programadas e não completadas, % (PEMOB)",
    "onibus_operacionais_por_100mil_hab": "Ônibus operacionais por 100 mil hab.",
    "pass_equiv_por_hab_ano": "Passageiros equivalentes por habitante no ano",
    "km_onibus_por_hab_ano": "Km rodados pelos ônibus por habitante no ano",
    "pass_equiv_por_onibus_dia": "Passageiros equivalentes por ônibus por dia",
    # deslocamento casa-trabalho (Censos)
    "pct_onibus": "% dos trabalhadores que vão de ônibus (Censo 2022)", "pct_brt": "% que vão de BRT (Censo 2022)",
    "pct_trem_metro": "% que vão de trem ou metrô (Censo 2022)", "pct_automovel": "% que vão de automóvel (Censo 2022)",
    "pct_moto": "% que vão de moto (Censo 2022)", "pct_a_pe": "% que vão a pé (Censo 2022)",
    "pct_bicicleta": "% que vão de bicicleta (Censo 2022)",
    "pct_coletivo": "% em transporte coletivo: ônibus + BRT + trem/metrô + van (Censo 2022)",
    "pct_individual_motorizado": "% em automóvel + moto + mototáxi + táxi (Censo 2022)",
    "pct_ativo": "% a pé ou de bicicleta (Censo 2022)",
    "trabalhadores_que_se_deslocam": "Trabalhadores que saem de casa para trabalhar (Censo 2022)",
    "pct_2010_ate_30min": "% com viagem de até 30 min (Censo 2010)", "pct_2010_30min_1h": "% de 30 min a 1 h (Censo 2010)",
    "pct_2010_1h_2h": "% de 1 h a 2 h (Censo 2010)", "pct_2010_mais_2h": "% com mais de 2 h (Censo 2010)",
    "pct_2022_ate_30min": "% com viagem de até 30 min (Censo 2022)", "pct_2022_30min_1h": "% de 30 min a 1 h (Censo 2022)",
    "pct_2022_1h_2h": "% de 1 h a 2 h (Censo 2022)", "pct_2022_mais_2h": "% com mais de 2 h (Censo 2022)",
    "tempo_medio_2022_min": "Tempo médio casa-trabalho, min (ponto médio das faixas, Censo 2022)",
    "pct_mais_1h_2010": "% com viagem de mais de 1 h (Censo 2010)", "pct_mais_1h_2022": "% com viagem de mais de 1 h (Censo 2022)",
}
dicionario = []
for nome in ["base_salvador_anual", "base_salvador_mensal", "painel_capitais_anual", "pemob_capitais", "deslocamento_capitais_censo"]:
    for col, tipo in TABELAS[nome].dtypes.items():
        dicionario.append({"tabela": nome, "coluna": col, "tipo": str(tipo), "descricao": DESCRICOES.get(col, "ver notebook/relatório")})
dicionario = pd.DataFrame(dicionario)
dicionario.to_csv(PROCESSADOS / "dicionario_dados.csv", index=False, encoding="utf-8")
print(f"Dicionário com {len(dicionario)} colunas documentadas. ETL concluído.")

Dicionário com 111 colunas documentadas. ETL concluído.
